In [1]:
!pip install ftfy
!pip install transformers
!pip install tensorflow_addons
!pip install vncorenlp
!pip install bpemb


In [2]:
from google.colab import drive
import numpy as np
import pickle
import tensorflow as tf
import random
import tensorflow_addons as tfa
import pandas as pd
import warnings,operator
from tqdm import tqdm
from ftfy import fix_text
import os, re, string

warnings.filterwarnings("ignore")
from transformers import logging
from transformers import AutoTokenizer, AutoModel,AutoConfig, TFAutoModel, XLMRobertaTokenizer,XLMRobertaConfig,TFXLMRobertaModel
logging.set_verbosity_error()

from google.colab import drive
drive.mount('/content/gdrive')
path_train ='/content/gdrive/My Drive/2024_Research/J02. Data augmentation ABSA/Dataset/UIT_ABSA_Restaurant/csv/Train.csv'
path_dev ='/content/gdrive/My Drive/2024_Research/J02. Data augmentation ABSA/Dataset/UIT_ABSA_Restaurant/csv/Dev.csv'
path_test ='/content/gdrive/My Drive/2024_Research/J02. Data augmentation ABSA/Dataset/UIT_ABSA_Restaurant/csv/Test.csv'

path_test_original ='/content/gdrive/My Drive/2024_Research/J02. Data augmentation ABSA/Dataset/UIT_ABSA_Restaurant/Test.txt'


/usr/local/lib/python3.10/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [3]:
def set_seeds(seed=1):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)

def set_global_determinism(seed=1):
    set_seeds(seed=seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'


SEED = 42
set_global_determinism(seed=SEED)
tf.keras.utils.set_random_seed(SEED)

In [4]:
from vncorenlp import VnCoreNLP
rdrsegmenter = VnCoreNLP("/content/gdrive/MyDrive/2021_Research/Text Classification Research/vncorenlp/VnCoreNLP-1.1.1.jar", annotators="wseg", max_heap_size='-Xmx500m')
# Em có thể download file và tổ chức folder như đường dẫn này nhé:
# https://drive.google.com/drive/folders/1NGqMn3oCSXleKV-9TxH7NGctapZFqF1E?usp=drive_link

def word_segment(review):
  doc = rdrsegmenter.tokenize(review)
  doc = ' '.join([' '.join(x) for x in doc])
  doc = doc.translate(doc.maketrans('', '', string.punctuation.replace("_",""))).replace("giá _ tiền", "giá_tiền").replace("giátiền", "giá_tiền")
  return doc

print(word_segment("giá 45k giá ngon"))


giá 45k giá ngon


# Read dataset

In [5]:
import pandas as pd

train_df = pd.read_csv(path_train)
dev_df = pd.read_csv(path_dev)
test_df = pd.read_csv(path_test)

train_df.head()

,id,review,clean_review,output
0,1,Giá 53k size vừa.,giá giá tiền cỡ vừa .,"{DRINKS#PRICES, neutral}, {DRINKS#STYLE&OPTION..."
1,2,Nhưng nói chung cũng hơi đắt.,nhưng nói chung cũng hơi đắt .,"{RESTAURANT#PRICES, negative}"
2,3,Mình ăn rất hôi mùi dầu.,mình ăn rất hôi mùi dầu .,"{FOOD#QUALITY, negative}"
3,4,Mình ăn chưa baoh thấy mùi hôi hải sản.,mình ăn chưa bao giờ thấy mùi hôi hải sản .,"{FOOD#QUALITY, positive}"
4,5,3 dĩa vs 2 lon Revive mà có 190k thui(.,num dĩa với num lon revive mà có giá tiền thui...,"{RESTAURANT#PRICES, positive}"


In [6]:
def to_category_vector_aspect_polarity(label):
    vector = np.zeros(4).astype(np.float64)
    if label.strip() != '':
        if 'positive' in label:
            vector[1] = 1.0
        elif 'neutral' in label:
            vector[2] = 1.0
        elif 'negative' in label:
            vector[3] = 1.0
        else:
            vector[0] = 1.0
    else:
        vector[0] = 1.0
    return vector

import re
def create_output_aspect_polarity(labels,categories,list_output):
    for i,category in enumerate(categories):
        if category in labels:
            try:
                output = re.findall('('+category+',\s(positive|negative|neutral))',labels)[0][0]
            except:
                output = ''
            list_output[i].append(to_category_vector_aspect_polarity(output))
        else:
            list_output[i].append(to_category_vector_aspect_polarity(''))
    return list_output

In [7]:
listLabel = "RESTAURANT#GENERAL,SERVICE#GENERAL,FOOD#QUALITY,FOOD#STYLE&OPTIONS,DRINKS#STYLE&OPTIONS,DRINKS#PRICES,RESTAURANT#PRICES,RESTAURANT#MISCELLANEOUS,AMBIENCE#GENERAL,FOOD#PRICES,LOCATION#GENERAL,DRINKS#QUALITY"
categories = listLabel.split(',')

x_train_segment = []

output_train1 = []
output_train2 = []
output_train3 = []
output_train4 = []
output_train5 = []
output_train6 = []
output_train7 = []
output_train8 = []
output_train9 = []
output_train10 = []
output_train11 = []
output_train12 = []

list_output_train = list()
list_output_train.append(output_train1)
list_output_train.append(output_train2)
list_output_train.append(output_train3)
list_output_train.append(output_train4)
list_output_train.append(output_train5)
list_output_train.append(output_train6)
list_output_train.append(output_train7)
list_output_train.append(output_train8)
list_output_train.append(output_train9)
list_output_train.append(output_train10)
list_output_train.append(output_train11)
list_output_train.append(output_train12)

for (label, review) in zip(train_df["output"].tolist(), train_df["clean_review"].tolist()):
    x_train_segment.append(review)
    create_output_aspect_polarity(label, categories,list_output_train)


x_valid_segment = []
y_valid_category = []

output_val1 = []
output_val2 = []
output_val3 = []
output_val4 = []
output_val5 = []
output_val6 = []
output_val7 = []
output_val8 = []
output_val9 = []
output_val10 = []
output_val11 = []
output_val12 = []

list_output_val = list()
list_output_val.append(output_val1)
list_output_val.append(output_val2)
list_output_val.append(output_val3)
list_output_val.append(output_val4)
list_output_val.append(output_val5)
list_output_val.append(output_val6)
list_output_val.append(output_val7)
list_output_val.append(output_val8)
list_output_val.append(output_val9)
list_output_val.append(output_val10)
list_output_val.append(output_val11)
list_output_val.append(output_val12)

for (label, review) in zip(dev_df["output"].tolist(), dev_df["clean_review"].tolist()):
    x_valid_segment.append(review)
    create_output_aspect_polarity(label, categories,list_output_val)

x_test_segment = []
x_test_original = []
for (label, review) in zip(test_df["output"].tolist(), test_df["clean_review"].tolist()):
    x_test_segment.append(review)
    x_test_original.append(label)


with open(path_test,"r",encoding="utf8") as file:
  content = file.read()
  with open("grouth_true.txt","w",encoding="utf8") as file:
      file.write(content)

print(len(x_train_segment),len(list_output_train))
print(len(x_valid_segment),len(list_output_val))


7028 12
771 12


In [8]:
print(x_train_segment[0])
print(x_train_segment[220])
print(x_train_segment[100])
print(x_train_segment[-1])

giá giá tiền cỡ vừa .
quán bán lâu rồi , chỉ bảng hiệu nhỏ thôi mà sáng nào khách cũng nườm nượp .
đồ ăn thì phải nói quá bình thường .
mì udon cọng vừa ăn , có cải ăn kèm nữa .


# Classification model

In [9]:
from bpemb import BPEmb
word_embedding = BPEmb(lang="multi", vs=1000000, dim=300)
EMBEDDING_DIM = word_embedding['nhà'].shape[0]
print(EMBEDDING_DIM)

300


In [10]:
def find_subtoken(unfound, word2vec_model, mode='initial'):
    found_affix = None
    if mode == 'initial':
        chunk = len(unfound) - 1
        while chunk > 2:
            try:
                word2vec_model[unfound[:chunk]]
                found_affix = unfound[:chunk]
                break
            except:
                chunk -= 1
    elif mode == 'final':
        chunk = 1
        while len(unfound) - chunk > 2:
            try:
                word2vec_model[unfound[chunk:]]
                found_affix = unfound[chunk:]
                break
            except:
                chunk += 1
    return found_affix

def find_subtoken2(unfound,word2vec_model):
    array = unfound.split('_')
    vector = np.zeros(EMBEDDING_DIM)
    check = False
    if len(array) > 1:
       for word2 in array:
          try:
              vector += word2vec_model[word2]
              check = True
          except:
              continue
    if check ==  True:
        return vector
    else:
        return [0]


In [11]:
def generate_embedding(word_index, model_embedding,EMBEDDING_DIM):
    count6 = 0
    countNot6 = 0
    #embedding_matrix = np.zeros((len(word_index) + 1, EMBEDDING_DIM))
    embedding_matrix = np.asarray([np.random.uniform(-0.01,0.01,EMBEDDING_DIM) for _ in range((len(word_index) + 1))])
    list_oov = []
    word_is_trained = []
    for word, i in word_index.items():
        try:
            embedding_vector = model_embedding[word]
            word_is_trained.append(word)
        except:
            b = find_subtoken2(word,model_embedding)
            if len(b) > 1:
                embedding_vector = b
            else:
                prefix = find_subtoken(word, model_embedding, mode='initial')
                suffix = find_subtoken(word, model_embedding, mode='final')

                if prefix != None and suffix != None:
                    embedding_vector = model_embedding[prefix] + model_embedding[suffix]
                elif prefix != None and suffix == None:
                    embedding_vector = model_embedding[prefix]
                elif prefix == None and suffix != None:
                    embedding_vector= model_embedding[suffix]
                else:
                    list_oov.append(word)
                    countNot6 +=1
            continue
        if embedding_vector is not None:
            count6 +=1
            embedding_matrix[i] = embedding_vector

    print('Number of words in pre-train embedding: ' + str(count6))
    print('Number of words not in pre-train embedding: ' + str(countNot6))
    print(list_oov)
    return embedding_matrix,word_is_trained

In [12]:
xLengths_res = [len(x.split(' ')) for x in x_train_segment]
h_res = sorted(xLengths_res)  #sorted lengths
MAX_LEN =h_res[len(h_res)-1]
print(MAX_LEN)

tokenizer = tf.keras.preprocessing.text.Tokenizer(filters="", oov_token='<UNK>')
tokenizer.fit_on_texts(x_train_segment)
word_index = tokenizer.word_index
input_vocab_size = len(tokenizer.word_index) + 1
print("input_vocab_size:",input_vocab_size)
train_seqs=tokenizer.texts_to_sequences(x_train_segment)

train_ids = tf.keras.preprocessing.sequence.pad_sequences(train_seqs, maxlen=MAX_LEN, dtype="long", value=0, truncating="post", padding="post")

val_seqs=tokenizer.texts_to_sequences(x_valid_segment)
val_ids = tf.keras.preprocessing.sequence.pad_sequences(val_seqs, maxlen=MAX_LEN, dtype="long", value=0, truncating="post", padding="post")


embedding_matrix,word_is_trained = generate_embedding(word_index,word_embedding,EMBEDDING_DIM)
print(len(word_is_trained))

109
input_vocab_size: 4006
Number of words in pre-train embedding: 1408
Number of words not in pre-train embedding: 1588
['<UNK>', 'nên', 'khá', 'viên', 'thấy', 'hơi', 'tốt', 'lắm', 'này', 'luôn', 'lại', 'vừa', 'hơn', 'nữa', 'ngọt', 'sữa', 'rồi', 'sốt', 'dễ', 'thêm', 'sẽ', 'chỗ', 'mới', 'nào', 'khác', 'đẹp', 'hết', 'đó', 'vẫn', 'ngồi', 'mấy', 'thử', 'loại', 'ổn', 'thôi', 'tệ', 'cảm', 'béo', 'giòn', 'lâu', 'mềm', 'cơm', 'kiểu', 'lẩu', 'vậy', 'chứ', 'đậm', 'chắc', 'mùi', 'tới', 'cực', 'biết', 'hôm', 'kèm', 'mặn', 'đều', 'nằm', 'đủ', 'nhận', 'rẻ', 'tìm', 'biệt', 'chọn', 'nhé', 'ghé', 'lạ', 'đâu', 'khó', 'tôm', 'chả', 'bún', 'phô', 'kêu', 'rãi', 'sắc', 'vẻ', 'tối', 'đợi', 'phí', 'cửa', 'hỏi', 'muốn', 'nhạt', 'mực', 'khô', 'ngán', 'cỡ', 'đào', 'chịu', 'giữ', 'nổi', 'gửi', 'mắc', 'cứ', 'giảm', 'mái', 'xíu', 'lớn', 'mọi', 'đầy', 'vài', 'nhẹ', 'thật', 'đặt', 'chấm', 'nấm', 'mắt', 'vệ', 'lấy', 'bỏ', 'mở', 'hài', 'đậu', 'trưa', 'buổi', 'miễn', 'đắt', 'mỡ', 'trộn', 'tiện', 'đổi', 'chế', 'hồi', 'c

In [13]:
print(EMBEDDING_DIM)

300


In [14]:
# PhoBERT model
inputs_review  = tf.keras.layers.Input(shape=(MAX_LEN, ), dtype='float64')
embedding_layer_domain = tf.keras.layers.Embedding(input_vocab_size,300,input_length=MAX_LEN, trainable=True, weights=[embedding_matrix])(inputs_review)
embedding_layer_domain = tf.keras.layers.SpatialDropout1D(0.5)(embedding_layer_domain)

bilstm_layer = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(256,activation='relu',return_sequences=True))(embedding_layer_domain)

conv_1 = tf.keras.layers.Conv1D(128, 2, padding="same", activation="relu",kernel_initializer='he_normal',trainable=True)(bilstm_layer)
conv_2 = tf.keras.layers.Conv1D(128, 3, padding="same", activation="relu",kernel_initializer='he_normal',trainable=True)(bilstm_layer)
conv_3 = tf.keras.layers.Conv1D(128, 4, padding="same", activation="relu",kernel_initializer='he_normal',trainable=True)(bilstm_layer)

maxpool_1 = tf.keras.layers.GlobalMaxPooling1D()(conv_1)
avepool_1 = tf.keras.layers.GlobalAveragePooling1D()(conv_1)
v1_col = tf.keras.layers.Concatenate(axis=1)([maxpool_1, avepool_1])

maxpool_2 = tf.keras.layers.GlobalMaxPooling1D()(conv_2)
avepool_2 = tf.keras.layers.GlobalAveragePooling1D()(conv_2)
v2_col = tf.keras.layers.Concatenate(axis=1)([maxpool_2, avepool_2])

maxpool_3 = tf.keras.layers.GlobalMaxPooling1D()(conv_3)
avepool_3 = tf.keras.layers.GlobalAveragePooling1D()(conv_3)
v3_col = tf.keras.layers.Concatenate(axis=1)([maxpool_3, avepool_3])

cls_token = tf.keras.layers.Concatenate(axis=1)([v1_col, v2_col,v3_col])

output1 = tf.keras.layers.Dense(4,name="output1", activation='softmax')(cls_token)
output2 = tf.keras.layers.Dense(4,name="output2", activation='softmax')(cls_token)
output3 = tf.keras.layers.Dense(4,name="output3", activation='softmax')(cls_token)
output4 = tf.keras.layers.Dense(4,name="output4", activation='softmax')(cls_token)
output5 = tf.keras.layers.Dense(4,name="output5", activation='softmax')(cls_token)
output6 = tf.keras.layers.Dense(4,name="output6", activation='softmax')(cls_token)
output7 = tf.keras.layers.Dense(4,name="output7", activation='softmax')(cls_token)
output8 = tf.keras.layers.Dense(4,name="output8", activation='softmax')(cls_token)
output9 = tf.keras.layers.Dense(4,name="output9", activation='softmax')(cls_token)
output10 = tf.keras.layers.Dense(4,name="output10", activation='softmax')(cls_token)
output11 = tf.keras.layers.Dense(4,name="output11", activation='softmax')(cls_token)
output12 = tf.keras.layers.Dense(4,name="output12", activation='softmax')(cls_token)

model_bilstmcnn = tf.keras.Model(inputs=inputs_review, outputs=[output1,output2,output3,output4,output5,output6,output7,output8,output9,output10,output11,output12])

#opt = tf.keras.optimizers.Adam(learning_rate=0.01)
opt = tfa.optimizers.RectifiedAdam(lr=0.001)
loss = tf.keras.losses.CategoricalCrossentropy()
metric = tf.metrics.CategoricalAccuracy('accuracy')

model_bilstmcnn.compile(loss={'output1':loss,'output2':loss,'output3':loss,'output4':loss,'output5':loss,'output6':loss,'output7':loss,'output8':loss,'output9':loss,'output10':loss,'output11':loss,'output12':loss},optimizer=opt, metrics = [metric])

list_total_Y_train  = [np.array(list_output_train[0]),np.array(list_output_train[1]),np.array(list_output_train[2]),np.array(list_output_train[3]),np.array(list_output_train[4]),np.array(list_output_train[5]),np.array(list_output_train[6]),np.array(list_output_train[7]),np.array(list_output_train[8]),np.array(list_output_train[9]),np.array(list_output_train[10]),np.array(list_output_train[11])]
list_total_Y_val  = [np.array(list_output_val[0]),np.array(list_output_val[1]),np.array(list_output_val[2]),np.array(list_output_val[3]),np.array(list_output_val[4]),np.array(list_output_val[5]),np.array(list_output_val[6]),np.array(list_output_val[7]),np.array(list_output_val[8]),np.array(list_output_val[9]),np.array(list_output_val[10]),np.array(list_output_val[11])]

history = model_bilstmcnn.fit(train_ids, list_total_Y_train, validation_data=(val_ids,list_total_Y_val) ,batch_size=32, epochs=100)
print(model_bilstmcnn.summary())

Epoch 1/100
220/220 [==============================] - 134s 558ms/step - loss: 136.6009 - output1_loss: 0.5971 - output2_loss: 0.6251 - output3_loss: 0.8375 - output4_loss: 131.1384 - output5_loss: 0.4226 - output6_loss: 0.2709 - output7_loss: 0.4132 - output8_loss: 0.4539 - output9_loss: 0.5387 - output10_loss: 0.3752 - output11_loss: 0.4014 - output12_loss: 0.5269 - output1_accuracy: 0.8537 - output2_accuracy: 0.8445 - output3_accuracy: 0.7048 - output4_accuracy: 0.7214 - output5_accuracy: 0.8963 - output6_accuracy: 0.9454 - output7_accuracy: 0.9244 - output8_accuracy: 0.9042 - output9_accuracy: 0.8692 - output10_accuracy: 0.9415 - output11_accuracy: 0.8907 - output12_accuracy: 0.8714 - val_loss: 4.9277 - val_output1_loss: 0.5124 - val_output2_loss: 0.5185 - val_output3_loss: 0.6758 - val_output4_loss: 0.7617 - val_output5_loss: 0.3104 - val_output6_loss: 0.1535 - val_output7_loss: 0.2799 - val_output8_loss: 0.3473 - val_output9_loss: 0.4400 - val_output10_loss: 0.2452 - val_output11

KeyboardInterrupt: ignored

In [ ]:
test_seqs=tokenizer.texts_to_sequences(x_test_segment)
test_ids = tf.keras.preprocessing.sequence.pad_sequences(test_seqs, maxlen=MAX_LEN, dtype="long", value=0, truncating="post", padding="post")


In [ ]:
import operator
textPrint =  ""
count = 0
for index,item in enumerate(test_ids):
    predicted = model_bilstmcnn.predict(np.expand_dims(item, axis=0))
    s = ''
    for i, predict in enumerate(predicted):
        index2, value = max(enumerate(predict[0]), key=operator.itemgetter(1))
        if index2 == 1:
            s+= '{' + str(categories[i]) + ', positive}, '
        elif index2 == 2:
            s+= '{' + str(categories[i]) + ', neutral}, '
        elif index2 == 3:
            s+= '{' + str(categories[i]) + ', negative}, '
    if s.strip() == "":
      s = "{RESTAURANT#GENERAL, neutral}, "
    textPrint += '#' + str(count +1)+'\n'
    textPrint += x_test_original[index] + '\n'
    textPrint += s[:len(s)-2] +'\n\n'
    count +=1
textPrint = textPrint[:-2]
with open('output_BiLSTM_CNN.txt','w',encoding = 'utf8') as file:
    file.write(textPrint)
print("Done")

# Evaluation

In [ ]:
import re
import sys

def get_labels_from_filename(filename):
    labels = []
    with open(filename, 'r', encoding = 'utf-8') as file:
        datasets = file.read()
        count = 0
        for line in datasets.split('\n'):
            if line != '':
                if count == 0:
                    count += 1
                elif count == 1:
                    count += 1
                elif count == 2:
                    labels.append(line.strip())
                    count = 0
        file.close()
    #print(len(labels))
    return labels

def clean_label(label):
    label = re.sub('[^A-Za-z#&]', '', label)
    label = re.sub('\\s+', ' ', label)
    return label

def convert_labels_to_dict(labels):
    dict_labels = []
    for label in labels:
        label_line = label.split('},')
        _dict = {}
        for objectLabel in label_line:
            aspect = clean_label(objectLabel.split(',')[0]).strip()
            polarity = clean_label(objectLabel.split(',')[1]).strip()
            _dict[aspect] = polarity
        dict_labels.append(_dict)
    return dict_labels

def get_common_attributeEntities(dict_labels):
    AttributeEntities = []
    for _dict in dict_labels:
        for key in _dict:
            if key not in AttributeEntities:
                AttributeEntities.append(key)
    AttributeEntities = sorted(AttributeEntities)
    return AttributeEntities

def get_aspects(dict_labels):
    aspects = []
    for _dict in dict_labels:
        for key in _dict:
            aspects.append(key)
    return aspects

def count_aspects(labels, Common_AttributeEntities):
    aspects = get_aspects(labels)
    num_aspects = [0] * len(Common_AttributeEntities)
    for aspect in aspects:
        num_aspects[Common_AttributeEntities.index(aspect)] += 1
    return num_aspects

def evaluation_labels(gold_labels, answer_labels, Common_AttributeEntities):
    num_aspect_gold = count_aspects(gold_labels, Common_AttributeEntities)
    num_aspect_answer = count_aspects(answer_labels, Common_AttributeEntities)
    correct_answer_aspects = [0] * len(Common_AttributeEntities)
    correct_answer_labels = [0] * len(Common_AttributeEntities)

    for i, _dict in enumerate(answer_labels):
        for key in _dict:
            if key in gold_labels[i].keys():
                correct_answer_aspects[Common_AttributeEntities.index(key)] += 1
                if answer_labels[i][key].strip() == gold_labels[i][key].strip():
                    correct_answer_labels[Common_AttributeEntities.index(key)] += 1
    #print('Correct Answer Aspects: ', correct_answer_aspects)
    #print('---------------------------------------------------')
    #print('Correct Answer Labels: ', correct_answer_labels)
    #print('---------------------------------------------------')
    #infor_evaluation(correct_answer_aspects, num_aspect_answer, num_aspect_gold, Common_AttributeEntities)
    #print('---------------------------------------------------')
    infor_evaluation(correct_answer_labels, num_aspect_answer, num_aspect_gold, Common_AttributeEntities)

def infor_evaluation(correct_answer, num_aspect_answer, num_aspect_gold, Common_AttributeEntities):
    for aspect in Common_AttributeEntities:
        if correct_answer[Common_AttributeEntities.index(aspect)] == 0:
            p = r = f = 0.0
        else:
            p = correct_answer[Common_AttributeEntities.index(aspect)] * 100 / num_aspect_answer[Common_AttributeEntities.index(aspect)]
            r = correct_answer[Common_AttributeEntities.index(aspect)] * 100 / num_aspect_gold[Common_AttributeEntities.index(aspect)]
            f = 2 * p * r / (p + r)
        print(aspect)
        print('%0.2f\t%0.2f\t%0.2f' % (p, r, f))
    p = sum(correct_answer) * 100 / sum(num_aspect_answer)
    r = sum(correct_answer) * 100 / sum(num_aspect_gold)
    f = 2 * p * r / (p + r)
    print('-------------------------------------------------------------')
    print('-------------------------------------------------------------')
    print('Mean Precision score: ', round(p,2))
    print('Mean Recall score: ', round(r,2))
    print('Mean F1 score: ', round(f,2))
    print('-------------------------------------------------------------')
    print('-------------------------------------------------------------')

def evaluation_system(gold_labels, answer_labels):
    gold_dicts = convert_labels_to_dict(gold_labels)
    answer_dicts = convert_labels_to_dict(answer_labels)
    AttributeEntities = get_common_attributeEntities(gold_dicts)
    #print('---------------INFORMATION FILE--------------------')
    #print('Aspect Name: ', AttributeEntities)
    #print("Aspect Gold: ", count_aspects(gold_dicts, AttributeEntities))
    #print("Aspect Answer: ", count_aspects(answer_dicts, AttributeEntities))
    #print('---------------------------------------------------')
    evaluation_labels(gold_dicts, answer_dicts, AttributeEntities)

def evaluation_system_by_file(file_gold, file_predict):
    gold_labels = get_labels_from_filename(file_gold)
    answer_labels = get_labels_from_filename(file_predict)
    evaluation_system(gold_labels, answer_labels)

In [ ]:
# Caculate score of phoBERT
print("Score of PhoBERT model")
evaluation_system_by_file(path_test_original, "output_BiLSTM_CNN.txt")